Model1

In [ ]:
import torch
import torch.nn as nn
import numpy as np

In [ ]:
# Check for available device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# If GPU is available, print its properties
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Capability: {torch.cuda.get_device_capability(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print(f"Memory Cached: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")
    print(f"Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**2:.2f} MB")
else:
    print("No GPU available, using CPU.")

In [ ]:
class VocabularyEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(VocabularyEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, input_indices):
        # input_indices: Tensor of shape (batch_size, sequence_length)
        embedded = self.embedding(input_indices)
        # embedded: Tensor of shape (batch_size, sequence_length, embedding_dim)
        return embedded

In [ ]:
class GraphTransformerEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_heads, num_layers):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=num_heads, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, X, A):
        # Optionally add positional/structural encodings here
        X_proj = self.input_proj(X)
        return self.transformer(X_proj)  # returns node embeddings H ∈ ℝ^{N×D}


In [ ]:
class NewNodeEmbedding(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.context_pool = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, H):
        context = H.mean(dim=1)  # Global graph context
        return self.context_pool(context).unsqueeze(1)  # Shape: (B, 1, D)


In [ ]:
class EdgePredictor(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.edge_score = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, h_existing, h_new):
        N = h_existing.size(1)
        h_new_expand = h_new.expand(-1, N, -1)  # (B, N, D)
        edge_input = torch.cat([h_existing, h_new_expand], dim=-1)
        edge_logits = self.edge_score(edge_input).squeeze(-1)  # (B, N)
        return torch.sigmoid(edge_logits)


In [ ]:
class GraphGrowNet(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, num_heads=4, num_layers=3):
        super().__init__()
        self.encoder = GraphTransformerEncoder(in_dim, hidden_dim, num_heads, num_layers)
        self.node_gen = NewNodeEmbedding(hidden_dim)
        self.edge_pred = EdgePredictor(hidden_dim)

    def forward(self, X, A):
        H = self.encoder(X, A)  # (B, N, D)
        h_new = self.node_gen(H)  # (B, 1, D)
        edges = self.edge_pred(H, h_new)  # (B, N)
        return edges  # predicted adjacency row for new node


In [ ]:
from torch.utils.data import Dataset, DataLoader

class CircuitGraphDataset(Dataset):
    def __init__(self, data_list):
        """
        data_list: list of tuples (adj_matrix, node_features, target_edges)
        """
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        A, X, y = self.data[idx]
        return {
            "adj": torch.tensor(A, dtype=torch.float),
            "features": torch.tensor(X, dtype=torch.float),
            "target": torch.tensor(y, dtype=torch.float),
        }

def collate_fn(batch):
    return {
        "adj": [item["adj"] for item in batch],
        "features": [item["features"] for item in batch],
        "target": [item["target"] for item in batch],
    }



In [ ]:
from Circuits import Circuits
circuits=Circuits()
graphdataset,textdataset=circuits.data_lodder_withoutpading()

In [ ]:
dataset[0]["features"]

In [ ]:
import torch.nn as nn
import torch.optim as optim

def pad_batch(batch):
    """Pads adjacency and feature matrices to match max size in batch."""
    max_nodes = max([adj.size(0) for adj in batch["adj"]])
    padded_adj, padded_feat, padded_target = [], [], []

    for A, X, y in zip(batch["adj"], batch["features"], batch["target"]):
        # F = 1
        N, N = A.shape
        pad_A = torch.zeros((max_nodes, max_nodes))
        # pad_X = torch.zeros((max_nodes, F))
        pad_X = torch.zeros((max_nodes))
        pad_y = torch.zeros((max_nodes))
        print(X.shape)
        pad_A[:N, :N] = A
        #pad_X[:N, :] = X
        pad_X[:N] = X
        pad_y[:N] = y
        print(np.matrix(pad_X).shape)

        padded_adj.append(pad_A)
        padded_feat.append(pad_X)
        padded_target.append(pad_y)

    return {
        "adj": torch.stack(padded_adj),
        "features": torch.stack(padded_feat),
        "target": torch.stack(padded_target),
    }

def train(model, dataloader, epochs=10, lr=1e-3, device="cuda"):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in dataloader:
            batch = pad_batch(batch)
            A, X, y = batch["adj"].to(device), batch["features"].to(device), batch["target"].to(device)

            optimizer.zero_grad()
            preds = model(X, A)  # shape (B, N)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"[Epoch {epoch+1}/{epochs}] Loss: {total_loss:.4f}")


In [ ]:
n,d=8,8
X = torch.rand(n, d)
A = torch.randint(0, 2, (n, n)).float()
A = torch.triu(A, diagonal=1)
A = A + A.T  # make symmetric
y = torch.randint(0, 2, (n,)).float()  # ground truth new adjacency row

In [ ]:
# Generate train_data from graphdataset and textdataset
train_data = []

for i, graph in enumerate(graphdataset):
    # Adjacency matrix
    adj_matrix = graph.numpy()

    
    # Node features from textdataset
    node_features = textdataset[i].numpy()
    
    # Target ground truth connections (for simplicity, using the adjacency matrix as the target)
    target_edges = adj_matrix[-1].copy()
    
    # Append to train_data
    train_data.append((adj_matrix, node_features, target_edges))

print(f"Generated train_data with {len(train_data)} samples.")

In [ ]:
train_data[0][0].__len__(),train_data[0][1].__len__()

In [ ]:

# Add more samples as needed

dataset = CircuitGraphDataset(train_data)
loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

model = GraphGrowNet(in_dim=node_features_1.shape[1]).to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters in the model: {num_params}")
train(model, loader, epochs=20)


In [ ]:
def remove_node(adj_matrix, node_index):
    """
    Removes the specified node (row and column) from the adjacency matrix.

    Parameters:
    adj_matrix (torch.Tensor): The adjacency matrix.
    node_index (int): The index of the node to remove.

    Returns:
    torch.Tensor: The updated adjacency matrix.
    """
    updated_adj_matrix = torch.cat((adj_matrix[:node_index], adj_matrix[node_index+1:]), dim=0)
    
    updated_adj_matrix = torch.cat((updated_adj_matrix[:, :node_index], updated_adj_matrix[:, node_index+1:]), dim=1)
    return updated_adj_matrix

# Example usage
node_to_remove =len(A)-1
 # Specify the node index to remove
updated_adj_matrix = remove_node(A, node_to_remove)
print(updated_adj_matrix)

In [ ]:
losses = []

# Iterate through each node (row) in the adjacency matrix
for node_index in range(A.size(0)):
    # Remove the specified node (row and column) from the adjacency matrix
    updated_adj_matrix = remove_node(A, node_index)
    
    # Remove the corresponding node features
    updated_features = torch.cat((X[:node_index], X[node_index+1:]), dim=0)
    
    # Remove the corresponding target edges
    updated_target = torch.cat((y[:node_index], y[node_index+1:]), dim=0)
    
    # Pass the updated adjacency matrix and features through the model
    pred = model(updated_features, updated_adj_matrix)
    
    # Compute the loss
    loss = loss_fn(pred, updated_target)
    losses.append(loss.item())

    print(f"Node {node_index} removed, Loss: {loss.item():.4f}")

# Print the average loss
avg_loss = sum(losses) / len(losses)
print(f"Average Loss after removing nodes: {avg_loss:.4f}")

In [ ]:
# Training loop for adjacency matrix with n-1 components
num_epochs = 10
losses = []

for epoch in range(num_epochs):
    total_loss = 0.0
    for i in range(A.size(0)):  # Iterate over each node
        # Remove the i-th node from the adjacency matrix and features
        adj_n_minus_1 = remove_node(A, i)
        features_n_minus_1 = torch.cat((X[:i], X[i+1:]), dim=0)
        
        # Use the i-th row of the adjacency matrix as the target
        target_row = A[i]
        
        # Pass the reduced adjacency matrix and features through the model
        pred_row = model(features_n_minus_1, adj_n_minus_1)
        
        # Compute the loss
        loss = loss_fn(pred_row, target_row[:target_row.size(0) - 1])  # Exclude self-loop
        losses.append(loss.item())
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / A.size(0)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")